## Translation Assistant

In [0]:
# Install required packages
%pip install langchain langchain-core mlflow openai --quiet
dbutils.library.restartPython()

In [0]:
import mlflow
from langchain_core.prompts import PromptTemplate

# Use MLflow to create a langchain-compatible LLM wrapper
class DatabricksLLM:
    def __init__(self, endpoint_name):
        self.endpoint_name = endpoint_name
    
    def invoke(self, prompt):
        """Invoke the Databricks Foundation Model endpoint."""
        from mlflow.deployments import get_deploy_client
        
        client = get_deploy_client("databricks")
        response = client.predict(
            endpoint=self.endpoint_name,
            inputs={
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.1,
                "max_tokens": 1000
            }
        )
        return response['choices'][0]['message']['content']

# Initialize the LLM with a Foundation Model endpoint
llm = DatabricksLLM(endpoint_name="databricks-meta-llama-3-3-70b-instruct")

# Create a translation prompt template with clear instructions
translation_prompt = PromptTemplate(
    input_variables=["text", "target_language"],
    template="""You are a professional translator with expertise in multiple languages.

Your task: Translate the following text to {target_language}.

Important instructions:
- Provide ONLY the translation
- Maintain the tone and style of the original text
- Do not add explanations or comments
- Preserve any formatting or punctuation

Text to translate:
{text}

Translation:"""
)

print("✓ Translation components initialized successfully!")

In [0]:
def translate_text(text: str, target_language: str) -> str:
    """
    Translate text to the target language using the LLM and prompt template.
    
    Args:
        text: The text to translate
        target_language: The target language (e.g., 'Spanish', 'French', 'German', 'Japanese')
    
    Returns:
        Translated text as a string
    """
    # Format the prompt with the input variables
    prompt = translation_prompt.format(
        text=text,
        target_language=target_language
    )
    
    # Invoke the LLM with the formatted prompt
    result = llm.invoke(prompt)
    return result.strip()

# Example 1: English to Spanish
print("=" * 60)
print("Example 1: English → Spanish")
print("=" * 60)
text1 = "Hello, how are you today? I hope you're having a great day!"
print(f"Original: {text1}")
translation1 = translate_text(text1, "Spanish")
print(f"Translation: {translation1}")

# Example 2: English to French
print("\n" + "=" * 60)
print("Example 2: English → French")
print("=" * 60)
text2 = "The quick brown fox jumps over the lazy dog."
print(f"Original: {text2}")
translation2 = translate_text(text2, "French")
print(f"Translation: {translation2}")

# Example 3: English to German
print("\n" + "=" * 60)
print("Example 3: English → German")
print("=" * 60)
text3 = "Data science is transforming the way we make decisions."
print(f"Original: {text3}")
translation3 = translate_text(text3, "German")
print(f"Translation: {translation3}")

print("\n" + "=" * 60)
print("✓ Translation examples completed!")
print("\nYou can now use translate_text(text, target_language) for any translation.")

In [0]:
# Remove existing widgets if any
dbutils.widgets.removeAll()

# Create input widgets
dbutils.widgets.text("input_text", "Hello, welcome to Databricks!", "Enter text to translate:")

dbutils.widgets.dropdown(
    "target_lang",
    "Spanish",
    ["Spanish", "French", "German", "Italian", "Portuguese", "Japanese", "Chinese", "Korean", "Russian", "Arabic", "Hindi", "Malayalam"],
    "Select target language:"
)

print("✓ Interactive widgets created!")
print("\n" + "=" * 60)


In [0]:
# Get values from widgets
user_text = dbutils.widgets.get("input_text")
selected_language = dbutils.widgets.get("target_lang")

# Display input
print("=" * 70)
print("INPUT")
print("=" * 70)
print(f"Original Text: {user_text}")
print(f"Target Language: {selected_language}")
print()

# Perform translation
print("=" * 70)
print("TRANSLATION RESULT")
print("=" * 70)
translation_result = translate_text(user_text, selected_language)
print(f"\n{translation_result}\n")
print("=" * 70)
print("✓ Translation completed successfully!")